# FlightRisk — Exploratory Data Analysis

Quick exploration of the (synthetic sample or real BTS) flight on-time performance data used to train FlightRisk.

This notebook is for exploration only — the actual production pipeline lives in `src/` and is run via the scripts in `scripts/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import SAMPLE_CSV_PATH
from src.data.clean import clean_flights
from src.data.load_data import normalize_columns
from src.features.build_features import add_schedule_features

pd.set_option('display.max_columns', 50)

In [ ]:
raw_df = pd.read_csv(SAMPLE_CSV_PATH)
raw_df = normalize_columns(raw_df)
print(raw_df.shape)
raw_df.head()

In [ ]:
clean_df = clean_flights(raw_df)
print(clean_df.shape)
clean_df['ArrDel15'].value_counts(normalize=True)

## Delay rate by scheduled departure hour

In [ ]:
feat_df = add_schedule_features(clean_df)
hourly = feat_df.groupby('DepHour')['ArrDel15'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(kind='bar', ax=ax)
ax.set_xlabel('Scheduled departure hour')
ax.set_ylabel('Delay rate (ArrDel15 == 1)')
ax.set_title('Delay rate by scheduled departure hour')
plt.tight_layout()
plt.show()

## Delay rate by carrier and by route

In [ ]:
carrier_rates = feat_df.groupby('Airline')['ArrDel15'].mean().sort_values(ascending=False)
carrier_rates

In [ ]:
route_rates = feat_df.groupby('Route')['ArrDel15'].agg(['mean', 'count'])
route_rates[route_rates['count'] >= 10].sort_values('mean', ascending=False).head(10)

## Next steps

See `scripts/prepare_data.py`, `scripts/train_model.py`, and `scripts/evaluate_model.py` for the full production pipeline that turns this exploration into a trained, evaluated, and served model.